# 🔥 Pipeline v2: IA de Gravedad de Incendios (Refinado)
## Municipalidad Valle del Sol

### ⚡ Mejoras v2 vs v1:
| v1 (anterior) | v2 (este notebook) |
|---|---|
| Etiquetado por color RGB básico | **CLIP (IA de OpenAI)** que entiende el contenido real |
| Solo detectaba pixeles rojos | **HSV color space** + análisis de textura |
| Solo incendios forestales | **Forestales + urbanos/domésticos** |
| 3 datasets | **5 datasets** (más variedad) |
| Sin validación | **Triple validación** (CLIP + HSV + textura) |

### ⚠️ Antes de empezar:
1. **Runtime → Change runtime type → T4 GPU** (obligatorio)
2. Cuenta de Kaggle + API Key (`kaggle.json`)
3. API Key NASA FIRMS (gratis, opcional para datos satelitales)

### 🚀 Ejecuta cada celda en orden (Shift+Enter)

---
## 📁 Paso 1: Conectar Google Drive + Instalar dependencias

In [ ]:
# ============================================================
# PASO 1: SETUP COMPLETO
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/fire_severity_project'

CARPETAS = [
    f'{BASE}/datasets/raw',
    f'{BASE}/datasets/cleaned/train/1_conato',
    f'{BASE}/datasets/cleaned/train/2_menor',
    f'{BASE}/datasets/cleaned/train/3_moderado',
    f'{BASE}/datasets/cleaned/train/4_mayor',
    f'{BASE}/datasets/cleaned/train/5_catastrofico',
    f'{BASE}/datasets/cleaned/val/1_conato',
    f'{BASE}/datasets/cleaned/val/2_menor',
    f'{BASE}/datasets/cleaned/val/3_moderado',
    f'{BASE}/datasets/cleaned/val/4_mayor',
    f'{BASE}/datasets/cleaned/val/5_catastrofico',
    f'{BASE}/datasets/tabular',
    f'{BASE}/models',
    f'{BASE}/reports',
]

for c in CARPETAS:
    os.makedirs(c, exist_ok=True)

# Instalar dependencias clave
print('📦 Instalando dependencias...')
!pip install -q transformers torch torchvision timm xgboost \
    onnxruntime Pillow scikit-learn kaggle joblib

print('\n✅ Google Drive conectado!')
print(f'📁 Proyecto en: {BASE}')

---
## 📥 Paso 2: Descargar datasets (forestales + urbanos/domésticos)

In [ ]:
# ============================================================
# PASO 2A: CONFIGURAR KAGGLE (YA CONFIGURADO)
# ============================================================
import os, json

# Credenciales ya configuradas - solo ejecuta esta celda
KAGGLE_USERNAME = 'ignaciosalazar'
KAGGLE_KEY = 'KGAT_234c293c3c421b434b0657794a247fcc'

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print(f'\u2705 Kaggle configurado para: {KAGGLE_USERNAME}')
print('   No necesitas subir ningun archivo .json')


In [ ]:
# ============================================================
# PASO 2B: DESCARGAR 5 DATASETS (forestales + urbanos)
# ============================================================
!pip install kaggle -q

print('='*65)
print('📥 DESCARGANDO DATASETS — Forestales + Urbanos/Domésticos')
print('='*65)

datasets_info = [
    # (slug de kaggle, carpeta destino, descripción)
    ('phylake1337/fire-dataset', '/content/ds/fire_dataset', '🌲 FIRE Dataset (forestal)'),
    ('brsdincer/wildfire-detection-image-data', '/content/ds/wildfire', '🌲 Wildfire Detection'),
    ('mohnishsaiprasad/forest-fire-images', '/content/ds/forest_fire', '🌲 Forest Fire Images'),
    ('atulyakumar98/test-dataset', '/content/ds/house_fire', '🏠 House Fire Dataset'),
    ('rituparna9/fire-detection-from-cctv', '/content/ds/cctv_fire', '🏠 CCTV Fire Detection'),
]

for i, (slug, dest, desc) in enumerate(datasets_info, 1):
    print(f'\n📦 [{i}/{len(datasets_info)}] {desc}...')
    try:
        !kaggle datasets download -d {slug} -p {dest} --unzip -q 2>/dev/null
        # Contar imágenes
        from pathlib import Path
        n = sum(1 for _ in Path(dest).rglob('*.jpg')) + sum(1 for _ in Path(dest).rglob('*.png'))
        print(f'   ✅ Listo! ({n:,} imágenes)')
    except Exception as e:
        print(f'   ⚠️  Error: {e} — continuando...')

print('\n' + '='*65)
print('✅ DESCARGA COMPLETADA')
print('='*65)

---
## 🧹 Paso 3: Limpieza de imágenes

In [ ]:
# ============================================================
# PASO 3: LIMPIEZA — eliminar corruptas, duplicadas, pequeñas
# ============================================================
from PIL import Image
import hashlib
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

def limpiar_dataset(input_dirs, output_dir, min_size=100):
    stats = Counter()
    hashes = set()
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    archivos = []
    for d in input_dirs:
        p = Path(d)
        if not p.exists():
            continue
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']:
            archivos.extend(list(p.rglob(ext)))

    print(f'🔍 {len(archivos):,} imágenes encontradas')

    for i, f in enumerate(archivos):
        stats['total'] += 1
        try:
            img = Image.open(f)
            img.verify()
            img = Image.open(f)
            img.load()
        except:
            stats['corruptas'] += 1
            continue

        w, h = img.size
        if w < min_size or h < min_size:
            stats['pequenas'] += 1
            continue

        try:
            thumb = img.copy()
            thumb.thumbnail((32, 32))
            h_val = hashlib.md5(thumb.tobytes()).hexdigest()
            if h_val in hashes:
                stats['duplicadas'] += 1
                continue
            hashes.add(h_val)
        except:
            continue

        try:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            nombre = f'img_{stats["ok"]:06d}.jpg'
            img.save(out / nombre, 'JPEG', quality=90)
            stats['ok'] += 1
        except:
            continue

        if (i+1) % 2000 == 0:
            print(f'  ⏳ {i+1:,}/{len(archivos):,} — guardadas: {stats["ok"]:,}')

    print(f'\n{"="*50}')
    print(f'📊 LIMPIEZA: {stats["total"]:,} → {stats["ok"]:,} imágenes')
    print(f'   ❌ Corruptas: {stats["corruptas"]} | Pequeñas: {stats["pequenas"]} | Duplicadas: {stats["duplicadas"]}')
    print(f'{"="*50}')
    return stats

stats = limpiar_dataset(
    input_dirs=['/content/ds/fire_dataset', '/content/ds/wildfire',
                '/content/ds/forest_fire', '/content/ds/house_fire',
                '/content/ds/cctv_fire'],
    output_dir='/content/imgs_clean',
    min_size=100
)

---
## 🧠 Paso 4: ETIQUETADO INTELIGENTE CON CLIP + HSV + TEXTURA

### ¿Por qué esto es MUCHO mejor que el método anterior?

| Método anterior (v1) | Método nuevo (v2) |
|---|---|
| Contaba píxeles rojos (RGB) | **CLIP de OpenAI** entiende QUÉ hay en la foto |
| Confundía atardeceres con fuego | **HSV** detecta fuego real por tono+saturación |
| No detectaba humo | **Análisis de textura** detecta humo y llamas |
| 1 sola señal | **3 señales combinadas** con pesos |

### ¿Qué es CLIP?
CLIP es una IA de OpenAI (gratis y open source) que puede mirar una imagen
y decirte qué tan parecida es a una descripción de texto. Ejemplo:
- Le das una foto y le preguntas: *"¿Esto se parece más a 'small fire' o a 'massive wildfire'?"*
- CLIP responde con porcentajes de similitud para cada opción.

In [ ]:
# ============================================================
# PASO 4A: CARGAR CLIP
# ============================================================
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

print('Cargando CLIP...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()
print('CLIP cargado!')

SEVERITY_PROMPTS = [
    ['a peaceful landscape with no fire',
     'a normal building with no fire or smoke',
     'a sunset or sunrise with orange sky but no fire',
     'red colored objects without any fire',
     'autumn leaves or fall colors'],
    ['a very small fire just starting, a tiny flame',
     'a small campfire or candle flame',
     'a small kitchen fire on a stove',
     'a small grass fire that just started',
     'a match or lighter flame, very small fire'],
    ['a localized fire burning a bush or small area',
     'a trash can or dumpster on fire',
     'a small room fire with visible flames',
     'a car fire or vehicle on fire',
     'a contained fire with some smoke'],
    ['a medium fire with significant flames and smoke',
     'a house room fully engulfed in flames',
     'a section of forest burning actively',
     'a building with fire coming out of windows',
     'a fire with thick dark smoke rising'],
    ['a large fire engulfing an entire building',
     'a large wildfire burning across hillside',
     'multiple structures on fire simultaneously',
     'a massive fire with huge flames and dense smoke',
     'firefighters battling a large blaze'],
    ['a catastrophic wildfire destroying everything',
     'an enormous out of control fire covering the entire landscape',
     'apocalyptic fire scene with sky turned red',
     'massive firestorm engulfing entire neighborhoods',
     'aerial view of enormous wildfire across vast area'],
]

def get_text_emb(texts):
    """Obtener embeddings de texto usando el modelo interno de CLIP."""
    tok = clip_processor.tokenizer(texts, return_tensors='pt',
                                   padding=True, truncation=True)
    with torch.no_grad():
        out = clip_model.text_model(
            input_ids=tok['input_ids'].to(DEVICE),
            attention_mask=tok['attention_mask'].to(DEVICE)
        )
        # Usar pooler_output y proyectar manualmente
        pooled = out.pooler_output  # (batch, 768)
        emb = clip_model.text_projection(pooled)  # (batch, 512)
    return emb

def get_image_emb(img):
    """Obtener embeddings de imagen usando el modelo interno de CLIP."""
    px = clip_processor.image_processor(images=img, return_tensors='pt')
    with torch.no_grad():
        out = clip_model.vision_model(
            pixel_values=px['pixel_values'].to(DEVICE)
        )
        pooled = out.pooler_output  # (1, 768)
        emb = clip_model.visual_projection(pooled)  # (1, 512)
    return emb

print('Pre-computando embeddings de texto...')
text_embeddings = []
for level_prompts in SEVERITY_PROMPTS:
    emb = get_text_emb(level_prompts)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    avg_emb = emb.mean(dim=0, keepdim=True)
    avg_emb = avg_emb / avg_emb.norm(dim=-1, keepdim=True)
    text_embeddings.append(avg_emb)

text_embeddings = torch.cat(text_embeddings, dim=0)
print(f'Listo! Shape: {text_embeddings.shape}')


In [ ]:
# ============================================================
# PASO 4B: FUNCIONES DE ANALISIS (CLIP + HSV + TEXTURA)
# ============================================================
import numpy as np
from PIL import Image, ImageFilter


def analisis_clip(img, text_embeddings):
    """Usa CLIP para entender QUE hay en la imagen."""
    img_emb = get_image_emb(img)
    img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
    similarities = (img_emb @ text_embeddings.T).squeeze(0).cpu().numpy()
    exp_sim = np.exp(similarities / 0.05)
    probs = exp_sim / exp_sim.sum()
    return probs


def analisis_hsv(img):
    img_small = img.resize((256, 256))
    pixels = np.array(img_small).astype(float) / 255.0
    r, g, b = pixels[:,:,0], pixels[:,:,1], pixels[:,:,2]
    total = r.shape[0] * r.shape[1]

    cmax = np.maximum(np.maximum(r, g), b)
    cmin = np.minimum(np.minimum(r, g), b)
    delta = cmax - cmin

    hue = np.zeros_like(r)
    mask_r = (cmax == r) & (delta > 0)
    mask_g = (cmax == g) & (delta > 0)
    mask_b = (cmax == b) & (delta > 0)
    hue[mask_r] = 60 * (((g[mask_r] - b[mask_r]) / delta[mask_r]) % 6)
    hue[mask_g] = 60 * (((b[mask_g] - r[mask_g]) / delta[mask_g]) + 2)
    hue[mask_b] = 60 * (((r[mask_b] - g[mask_b]) / delta[mask_b]) + 4)

    sat = np.where(cmax > 0, delta / cmax, 0)
    val = cmax

    fuego_rojo = ((hue < 15) | (hue > 345)) & (sat > 0.4) & (val > 0.4)
    fuego_naranja = (hue >= 15) & (hue <= 40) & (sat > 0.5) & (val > 0.5)
    fuego_amarillo = (hue > 40) & (hue <= 65) & (sat > 0.3) & (val > 0.6)
    fuego_blanco = (val > 0.85) & (sat < 0.2) & (r > 0.8) & (g > 0.6)
    brasas = ((hue < 20) | (hue > 340)) & (sat > 0.5) & (val > 0.15) & (val < 0.5)
    humo = (sat < 0.15) & (val > 0.25) & (val < 0.75)

    pct_fuego = (np.sum(fuego_rojo) + np.sum(fuego_naranja) +
                 np.sum(fuego_amarillo) + np.sum(fuego_blanco)) / total * 100
    pct_brasas = np.sum(brasas) / total * 100
    pct_humo = np.sum(humo) / total * 100
    mascara_fuego = fuego_rojo | fuego_naranja | fuego_amarillo | fuego_blanco
    intensidad = np.mean(val[mascara_fuego]) if np.any(mascara_fuego) else 0

    return {'pct_fuego': pct_fuego, 'pct_brasas': pct_brasas,
            'pct_humo': pct_humo, 'intensidad': intensidad}


def analisis_textura(img):
    from scipy import ndimage
    img_gray = img.convert('L').resize((128, 128))
    pixels = np.array(img_gray).astype(float)
    mean_local = ndimage.uniform_filter(pixels, size=5)
    var_local = ndimage.uniform_filter((pixels - mean_local)**2, size=5)
    gx = ndimage.sobel(pixels, axis=1)
    gy = ndimage.sobel(pixels, axis=0)
    return {'varianza_media': np.mean(var_local), 'varianza_max': np.max(var_local),
            'gradiente_medio': np.mean(np.sqrt(gx**2 + gy**2)), 'caos': np.std(var_local)}


def clasificar_imagen_v2(img_path, clip_model, clip_processor, text_embeddings):
    try:
        img = Image.open(img_path).convert('RGB')
    except:
        return 0, 0, {}

    clip_probs = analisis_clip(img, text_embeddings)
    clip_nivel = np.argmax(clip_probs)
    clip_conf = clip_probs[clip_nivel] * 100

    hsv = analisis_hsv(img)
    pct_total_fuego = hsv['pct_fuego'] + hsv['pct_brasas'] * 0.5
    if pct_total_fuego < 1: hsv_nivel = 0
    elif pct_total_fuego < 5: hsv_nivel = 1
    elif pct_total_fuego < 15: hsv_nivel = 2
    elif pct_total_fuego < 30: hsv_nivel = 3
    elif pct_total_fuego < 50: hsv_nivel = 4
    else: hsv_nivel = 5
    if hsv['pct_humo'] > 30 and hsv_nivel > 0:
        hsv_nivel = min(5, hsv_nivel + 1)

    tex = analisis_textura(img)
    es_textura_fuego = tex['caos'] > 500 and tex['varianza_media'] > 200
    tex_bonus = 1 if es_textura_fuego else 0

    score_final = (clip_nivel * 0.50 + hsv_nivel * 0.35 +
                   tex_bonus * 0.15 * max(clip_nivel, hsv_nivel))
    nivel_final = max(0, min(5, int(round(score_final))))

    if clip_probs[0] > 0.7 and hsv['pct_fuego'] < 3: nivel_final = 0
    if clip_nivel >= 4 and hsv_nivel >= 3: nivel_final = max(nivel_final, 4)

    acuerdo = 1.0 - abs(clip_nivel - hsv_nivel) / 5.0
    detalles = {
        'clip_nivel': clip_nivel, 'clip_conf': f'{clip_conf:.1f}%',
        'hsv_nivel': hsv_nivel, 'hsv_fuego': f'{pct_total_fuego:.1f}%',
        'hsv_humo': f'{hsv["pct_humo"]:.1f}%',
        'textura_fuego': es_textura_fuego,
        'acuerdo': f'{acuerdo:.0%}',
    }
    return nivel_final, acuerdo * 100, detalles


print('Funciones definidas!')


In [ ]:
# ============================================================
# PASO 4C: EJECUTAR ETIQUETADO INTELIGENTE
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'

from pathlib import Path
from collections import Counter
import shutil
import random
import time
random.seed(42)

CARPETAS = {
    1: '1_conato', 2: '2_menor', 3: '3_moderado',
    4: '4_mayor', 5: '5_catastrofico'
}
EMOJIS = {0: '🚫', 1: '🟢', 2: '🟡', 3: '🟠', 4: '🔴', 5: '⚫'}
DRIVE_CLEANED = f'{BASE}/datasets/cleaned'
SPLIT_RATIO = 0.85

# Limpiar datos anteriores si existen
for split in ['train', 'val']:
    for carpeta in CARPETAS.values():
        p = Path(DRIVE_CLEANED) / split / carpeta
        if p.exists():
            for f in p.glob('*.jpg'):
                f.unlink()

imagenes = list(Path('/content/imgs_clean').glob('*.jpg'))
random.shuffle(imagenes)
total = len(imagenes)
stats = Counter()
errores_detalle = []

print(f'🧠 Etiquetando {total:,} imágenes con CLIP + HSV + Textura')
print(f'   Esto toma ~2-5 minutos con GPU T4...\n')

start = time.time()

for i, img_path in enumerate(imagenes):
    nivel, confianza, detalles = clasificar_imagen_v2(
        img_path, clip_model, clip_processor, text_embeddings
    )

    if nivel == 0:
        stats['no_fuego'] += 1
        continue

    # Descartar si la confianza es muy baja (los métodos no se ponen de acuerdo)
    if confianza < 40:
        stats['baja_confianza'] += 1
        continue

    # Decidir split
    split = 'train' if random.random() < SPLIT_RATIO else 'val'

    # Guardar
    destino = Path(DRIVE_CLEANED) / split / CARPETAS[nivel]
    destino.mkdir(parents=True, exist_ok=True)
    nombre = f'fire_{nivel}_{stats[f"n{nivel}"]:05d}.jpg'

    try:
        img = Image.open(img_path).convert('RGB')
        img.save(destino / nombre, 'JPEG', quality=90)
        stats[f'n{nivel}'] += 1
        stats[split] += 1
    except:
        continue

    if (i+1) % 500 == 0:
        elapsed = time.time() - start
        eta = elapsed / (i+1) * (total - i - 1)
        print(f'  ⏳ {i+1:,}/{total:,} ({(i+1)/total*100:.0f}%) — '
              f'ETA: {eta/60:.1f}min — '
              f'Fuego: {sum(stats[f"n{n}"] for n in range(1,6)):,} | '
              f'Descartadas: {stats["no_fuego"]+stats["baja_confianza"]:,}')

elapsed = time.time() - start

print(f'\n{"="*60}')
print(f'🏷️  REPORTE DE ETIQUETADO INTELIGENTE')
print(f'{"="*60}')
print(f'  ⏱️  Tiempo: {elapsed/60:.1f} minutos')
print(f'  🚫 No es fuego (CLIP+HSV coinciden): {stats["no_fuego"]:,}')
print(f'  ❓ Baja confianza (descartadas):     {stats["baja_confianza"]:,}')
print(f'  ─────────────────────────────────')
for n in range(1, 6):
    print(f'  {EMOJIS[n]} Nivel {n}: {stats[f"n{n}"]:,}')
total_ok = sum(stats[f'n{n}'] for n in range(1, 6))
print(f'  ─────────────────────────────────')
print(f'  📊 Total etiquetadas: {total_ok:,}')
print(f'  📂 Train: {stats["train"]:,} | Val: {stats["val"]:,}')
print(f'{"="*60}')

---
## 👀 Paso 5: Revisión visual

In [ ]:
# ============================================================
# PASO 5: REVISIÓN VISUAL
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random

DRIVE_CLEANED = f'{BASE}/datasets/cleaned'
NIVELES = {
    '1_conato':       '🟢 Nivel 1: Conato',
    '2_menor':        '🟡 Nivel 2: Menor',
    '3_moderado':     '🟠 Nivel 3: Moderado',
    '4_mayor':        '🔴 Nivel 4: Mayor',
    '5_catastrofico': '⚫ Nivel 5: Catastrófico'
}

N = 8
fig, axes = plt.subplots(5, N, figsize=(2.5*N, 14))
fig.suptitle('👀 REVISIÓN VISUAL v2 — ¿Clasificación correcta?',
             fontsize=16, fontweight='bold', y=1.02)

for row, (carpeta, titulo) in enumerate(NIVELES.items()):
    ruta = Path(DRIVE_CLEANED) / 'train' / carpeta
    imgs = list(ruta.glob('*.jpg'))
    muestras = random.sample(imgs, min(N, len(imgs))) if imgs else []

    for col in range(N):
        if col < len(muestras):
            try:
                img = Image.open(muestras[col])
                axes[row][col].imshow(img)
            except:
                pass
        else:
            axes[row][col].text(0.5, 0.5, 'N/A', ha='center', va='center')
        axes[row][col].axis('off')

    axes[row][0].set_title(f'{titulo} ({len(imgs)} imgs)',
                           fontsize=9, fontweight='bold', loc='left')

plt.tight_layout()
plt.savefig(f'{BASE}/reports/revision_visual_v2.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'💾 Guardado en: {BASE}/reports/revision_visual_v2.png')

In [ ]:
# ============================================================
# PASO 5B: DIAGNÓSTICO DETALLADO — ver por qué se clasifica así
# ============================================================
import matplotlib.pyplot as plt
from pathlib import Path
import random

# Elegir 6 imágenes aleatorias y mostrar el razonamiento completo
all_imgs = []
for carpeta in NIVELES.keys():
    ruta = Path(DRIVE_CLEANED) / 'train' / carpeta
    all_imgs.extend(list(ruta.glob('*.jpg'))[:3])

random.shuffle(all_imgs)
muestras = all_imgs[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('🔬 DIAGNÓSTICO: Por qué se clasificó así', fontsize=14, fontweight='bold')

NOMBRES = {0:'No fuego', 1:'Conato', 2:'Menor', 3:'Moderado', 4:'Mayor', 5:'Catastrófico'}

for idx, img_path in enumerate(muestras):
    row, col = idx // 3, idx % 3
    nivel, conf, det = clasificar_imagen_v2(
        img_path, clip_model, clip_processor, text_embeddings
    )

    img = Image.open(img_path)
    axes[row][col].imshow(img)
    axes[row][col].set_title(
        f'{EMOJIS.get(nivel, "?")} Nivel {nivel}: {NOMBRES[nivel]}\n'
        f'CLIP→{det.get("clip_nivel","?")} | HSV→{det.get("hsv_nivel","?")} | '
        f'Fuego:{det.get("hsv_fuego","?")}\n'
        f'Acuerdo: {det.get("acuerdo","?")}',
        fontsize=8, fontweight='bold'
    )
    axes[row][col].axis('off')

plt.tight_layout()
plt.show()
print('\n📖 CLIP→ = lo que CLIP piensa | HSV→ = análisis de color | Fuego: = % de píxeles de fuego')

---
## ⚖️ Paso 6: Balancear clases

In [ ]:
# ============================================================
# PASO 6: BALANCEO CON DATA AUGMENTATION
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'
from PIL import Image, ImageEnhance
from pathlib import Path
import random

DRIVE_CLEANED = f'{BASE}/datasets/cleaned'
CARPETAS = ['1_conato', '2_menor', '3_moderado', '4_mayor', '5_catastrofico']
EMOJIS_LIST = ['🟢', '🟡', '🟠', '🔴', '⚫']

def augmentar(img):
    if random.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() > 0.7: img = img.transpose(Image.FLIP_TOP_BOTTOM)
    if random.random() > 0.5: img = img.rotate(random.randint(-20, 20), fillcolor=(0,0,0))
    if random.random() > 0.5: img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.4))
    if random.random() > 0.5: img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.3))
    return img

# Contar
conteos = {}
for c in CARPETAS:
    conteos[c] = len(list((Path(DRIVE_CLEANED)/'train'/c).glob('*.jpg')))

target = max(conteos.values())
if target == 0:
    print('⚠️ No hay imágenes. Revisa los pasos anteriores.')
else:
    print(f'📊 ANTES del balanceo:')
    for i, (c, n) in enumerate(conteos.items()):
        bar = '█' * max(1, n // 20)
        print(f'  {EMOJIS_LIST[i]} {c}: {n:,} {bar}')
    print(f'\n🎯 Objetivo: {target:,} por clase')

    for i, c in enumerate(CARPETAS):
        ruta = Path(DRIVE_CLEANED) / 'train' / c
        imgs = list(ruta.glob('*.jpg'))
        faltantes = target - len(imgs)
        if faltantes <= 0 or not imgs:
            continue
        print(f'  {EMOJIS_LIST[i]} {c}: generando {faltantes:,} imágenes...')
        for j in range(faltantes):
            try:
                base = Image.open(random.choice(imgs)).convert('RGB')
                aug = augmentar(base)
                aug.save(ruta / f'aug_{j:05d}.jpg', 'JPEG', quality=85)
            except:
                continue

    print(f'\n📊 DESPUÉS del balanceo:')
    for i, c in enumerate(CARPETAS):
        n = len(list((Path(DRIVE_CLEANED)/'train'/c).glob('*.jpg')))
        bar = '█' * max(1, n // 20)
        print(f'  {EMOJIS_LIST[i]} {c}: {n:,} {bar}')
    print('\n✅ Dataset balanceado!')

---
## 📊 Paso 7: Estadísticas finales

In [ ]:
# ============================================================
# PASO 7: ESTADÍSTICAS Y GRÁFICOS
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'
import matplotlib.pyplot as plt
from pathlib import Path

DRIVE_CLEANED = f'{BASE}/datasets/cleaned'
LABELS = ['Conato', 'Menor', 'Moderado', 'Mayor', 'Catastrófico']
COLORS = ['#22c55e', '#eab308', '#f97316', '#ef4444', '#1f2937']
CARPETAS = ['1_conato', '2_menor', '3_moderado', '4_mayor', '5_catastrofico']

train_c = [len(list((Path(DRIVE_CLEANED)/'train'/c).glob('*.jpg'))) for c in CARPETAS]
val_c = [len(list((Path(DRIVE_CLEANED)/'val'/c).glob('*.jpg'))) for c in CARPETAS]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars = ax1.bar(LABELS, train_c, color=COLORS, edgecolor='white', linewidth=2)
ax1.set_title('📂 Train', fontsize=13, fontweight='bold')
for b, c in zip(bars, train_c):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+5, f'{c:,}',
             ha='center', fontweight='bold')

bars = ax2.bar(LABELS, val_c, color=COLORS, edgecolor='white', linewidth=2)
ax2.set_title('📂 Validation', fontsize=13, fontweight='bold')
for b, c in zip(bars, val_c):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+2, f'{c:,}',
             ha='center', fontweight='bold')

plt.suptitle('🔥 Dataset Final v2 — Etiquetado con CLIP+HSV+Textura',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(f'{BASE}/reports/stats_v2.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'📊 Total: Train={sum(train_c):,} | Val={sum(val_c):,}')

---
## 🛰️ Paso 8: Datos NASA FIRMS

In [ ]:
# ============================================================
# PASO 8: DATOS SATELITALES NASA FIRMS
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'
import requests
import pandas as pd
from io import StringIO
import numpy as np

NASA_FIRMS_KEY = 'cd5eabfdf119f452966e8578df8e697f'
LAT_MIN, LAT_MAX = -34.0, -33.0
LON_MIN, LON_MAX = -71.5, -70.0

if NASA_FIRMS_KEY == 'cd5eabfdf119f452966e8578df8e697f':
    print('⚠️ Sin API Key de NASA FIRMS. Generando datos de ejemplo...')
    np.random.seed(42)
    n = 500
    df = pd.DataFrame({
        'latitude': np.random.uniform(LAT_MIN, LAT_MAX, n),
        'longitude': np.random.uniform(LON_MIN, LON_MAX, n),
        'brightness': np.random.uniform(300, 500, n),
        'bright_ti4': np.random.uniform(290, 450, n),
        'frp': np.random.exponential(50, n),
        'scan': np.random.uniform(0.3, 2.0, n),
        'track': np.random.uniform(0.3, 2.0, n),
        'confidence': np.random.choice(['low','nominal','high'], n, p=[0.2,0.5,0.3]),
        'daynight': np.random.choice(['D','N'], n),
    })
else:
    print('🛰️ Descargando datos reales de NASA FIRMS...')
    url = (f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/'
           f'{NASA_FIRMS_KEY}/VIIRS_SNPP_NRT/'
           f'{LON_MIN},{LAT_MIN},{LON_MAX},{LAT_MAX}/10')
    r = requests.get(url, timeout=30)
    df = pd.read_csv(StringIO(r.text)) if r.status_code == 200 else pd.DataFrame()

# Etiquetar gravedad
def etiquetar_firms(row):
    frp = row.get('frp', 0)
    bright = row.get('bright_ti4', row.get('brightness', 300))
    conf = row.get('confidence', 'nominal')
    if isinstance(conf, str):
        conf = {'low':30,'nominal':60,'high':90}.get(conf, 50)
    score = frp*0.35 + (bright-300)*0.25 + float(conf)*0.15 + row.get('scan',0.5)*row.get('track',0.5)*20
    if score > 200: return 5
    elif score > 120: return 4
    elif score > 60: return 3
    elif score > 25: return 2
    else: return 1

df['gravedad'] = df.apply(etiquetar_firms, axis=1)
df['area_estimada'] = df.get('scan', 0.5) * df.get('track', 0.5)

df.to_csv(f'{BASE}/datasets/tabular/firms_etiquetado.csv', index=False)
print(f'\n📊 {len(df):,} registros etiquetados')
print(df['gravedad'].value_counts().sort_index())

---
# 🧠 ENTRENAMIENTO DE MODELOS
---
## 🖼️ Paso 9: Entrenar CNN (EfficientNet-B0)
⚠️ Verificar: **Runtime → Change runtime type → T4 GPU**

In [ ]:
# ============================================================
# PASO 9A: SETUP CNN
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'
import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Dispositivo: {DEVICE}' +
      (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type=='cuda' else ' ⚠️ SIN GPU!'))

NUM_CLASSES = 5
BATCH_SIZE = 32
EPOCHS = 25
LR = 3e-4
DRIVE_CLEANED = f'{BASE}/datasets/cleaned'

# Data transforms
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(f'{DRIVE_CLEANED}/train', transform=train_tf)
val_ds = datasets.ImageFolder(f'{DRIVE_CLEANED}/val', transform=val_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'📂 Clases: {train_ds.classes}')
print(f'📊 Train: {len(train_ds):,} | Val: {len(val_ds):,}')

In [ ]:
# ============================================================
# PASO 9B: MODELO Y ENTRENAMIENTO
# ============================================================

BASE = '/content/drive/MyDrive/fire_severity_project'
class FireSeverityCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True)
        in_f = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_f, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.backbone(x)

model = FireSeverityCNN(NUM_CLASSES).to(DEVICE)

# Loss con pesos para penalizar más errores en incendios graves
weights = torch.tensor([1.0, 1.2, 1.5, 2.5, 4.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

history = {'tl':[], 'vl':[], 'ta':[], 'va':[]}
best_acc = 0.0
best_path = f'{BASE}/models/best_cnn_v2.pth'

print(f'🚀 Entrenando {EPOCHS} epochs...\n')
t0 = time.time()

for epoch in range(EPOCHS):
    # Train
    model.train()
    tl, tc, tt = 0, 0, 0
    for imgs, labs in train_dl:
        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labs)
        loss.backward()
        optimizer.step()
        tl += loss.item(); tc += out.argmax(1).eq(labs).sum().item(); tt += labs.size(0)

    # Val
    model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for imgs, labs in val_dl:
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            out = model(imgs)
            vl += criterion(out, labs).item()
            vc += out.argmax(1).eq(labs).sum().item(); vt += labs.size(0)

    ta = 100*tc/max(tt,1); va = 100*vc/max(vt,1)
    history['tl'].append(tl/max(len(train_dl),1))
    history['vl'].append(vl/max(len(val_dl),1))
    history['ta'].append(ta); history['va'].append(va)

    mark = ''
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), best_path)
        mark = ' ⭐'

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Loss {tl/max(len(train_dl),1):.3f}/{vl/max(len(val_dl),1):.3f} | '
          f'Acc {ta:.1f}%/{va:.1f}%{mark}')
    scheduler.step()

print(f'\n✅ Entrenamiento completado en {(time.time()-t0)/60:.1f} min')
print(f'🏆 Mejor Val Accuracy: {best_acc:.2f}%')
print(f'💾 Modelo guardado: {best_path}')

In [ ]:
# --- Gráficos de entrenamiento ---
BASE = '/content/drive/MyDrive/fire_severity_project'
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(history['tl'], 'b-', label='Train', lw=2)
a1.plot(history['vl'], 'r-', label='Val', lw=2)
a1.set_title('📉 Loss', fontsize=13, fontweight='bold')
a1.legend(); a1.grid(alpha=0.3)

a2.plot(history['ta'], 'b-', label='Train', lw=2)
a2.plot(history['va'], 'r-', label='Val', lw=2)
a2.axhline(best_acc, color='g', ls='--', label=f'Best: {best_acc:.1f}%')
a2.set_title('📈 Accuracy', fontsize=13, fontweight='bold')
a2.legend(); a2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/reports/training_v2.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 📊 Paso 10: Entrenar XGBoost

In [ ]:
# ============================================================
# PASO 10: XGBOOST
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pandas as pd, numpy as np, joblib
import matplotlib.pyplot as plt

df = pd.read_csv(f'{BASE}/datasets/tabular/firms_etiquetado.csv')
feats = [c for c in ['brightness','bright_ti4','frp','scan','track','area_estimada'] if c in df.columns]

if 'confidence' in df.columns and df['confidence'].dtype == 'object':
    df['conf_num'] = df['confidence'].map({'low':30,'nominal':60,'high':90}).fillna(50)
    feats.append('conf_num')
if 'daynight' in df.columns:
    df['noche'] = (df['daynight']=='N').astype(int)
    feats.append('noche')

X = df[feats].fillna(0); y = df['gravedad'] - 1
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss',
    random_state=42, verbosity=0)
m.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)

yp = m.predict(Xte)
acc = (yp==yte).mean()*100
print(f'🎯 XGBoost Accuracy: {acc:.2f}%\n')
print(classification_report(yte, yp, target_names=['Conato','Menor','Moderado','Mayor','Catastrófico'], zero_division=0))

xgb_path = f'{BASE}/models/fire_severity_xgboost_v2.joblib'
joblib.dump(m, xgb_path)
print(f'💾 Guardado: {xgb_path}')

# Confusion matrix
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(confusion_matrix(yte,yp),
    display_labels=['Conato','Menor','Moder','Mayor','Catastr']).plot(ax=ax, cmap='YlOrRd')
ax.set_title('XGBoost Confusion Matrix', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{BASE}/reports/cm_xgb_v2.png', dpi=120); plt.show()

---
## 📈 Paso 11: Evaluación CNN + Matriz de Confusión

In [ ]:
# ============================================================
# PASO 11: EVALUACIÓN CNN
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import numpy as np, matplotlib.pyplot as plt

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

preds, labels = [], []
with torch.no_grad():
    for imgs, labs in val_dl:
        out = model(imgs.to(DEVICE))
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(labs.numpy())

preds, labels = np.array(preds), np.array(labels)
cnn_acc = (preds==labels).mean()*100

print(f'🎯 CNN Accuracy: {cnn_acc:.2f}%\n')
names = ['Conato','Menor','Moderado','Mayor','Catastrófico']
print(classification_report(labels, preds, target_names=names, zero_division=0))

fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(confusion_matrix(labels, preds),
    display_labels=names).plot(ax=ax, cmap='YlOrRd')
ax.set_title('CNN Confusion Matrix', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.savefig(f'{BASE}/reports/cm_cnn_v2.png', dpi=120); plt.show()

print(f'\n🏆 RESUMEN: CNN={cnn_acc:.1f}% | XGBoost={acc:.1f}%')

---
## 🧪 Paso 12: Probar con imágenes de internet

In [ ]:
# ============================================================
# PASO 12: PRUEBA MANUAL CON FOTOS DE INTERNET
# ============================================================
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

NOMBRES = {0:'Conato', 1:'Menor', 2:'Moderado', 3:'Mayor', 4:'Catastrófico'}
EMOJIS_CNN = {0:'🟢', 1:'🟡', 2:'🟠', 3:'🔴', 4:'⚫'}

def probar_url(url, titulo=''):
    r = requests.get(url, timeout=10)
    img = Image.open(BytesIO(r.content)).convert('RGB')
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model(tensor)
        probs = torch.softmax(out, 1)[0]
        nivel = probs.argmax().item()
        conf = probs.max().item()*100
    print(f'{EMOJIS_CNN[nivel]} {NOMBRES[nivel]} ({conf:.1f}%) — {titulo}')
    return img, nivel, conf

# Probar con imágenes de ejemplo
urls = [
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/9/9a/Meadow_Fire.jpg/1280px-Meadow_Fire.jpg',
     'Incendio forestal grande'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/d/d8/Wildfire_in_California.jpg/1280px-Wildfire_in_California.jpg',
     'Wildfire California'),
]

fig, axes = plt.subplots(1, len(urls), figsize=(7*len(urls), 7))
if len(urls) == 1: axes = [axes]

for i, (url, titulo) in enumerate(urls):
    try:
        img, nivel, conf = probar_url(url, titulo)
        axes[i].imshow(img)
        axes[i].set_title(f'{EMOJIS_CNN[nivel]} {NOMBRES[nivel]} ({conf:.0f}%)\n{titulo}',
                         fontweight='bold')
        axes[i].axis('off')
    except Exception as e:
        print(f'⚠️ Error con {titulo}: {e}')

plt.tight_layout()
plt.show()

print('\n💡 Para probar con tus propias fotos, sube una a Colab y usa:')
print('   img, nivel, conf = probar_url(\'ruta/a/tu/imagen.jpg\')')

---
## 📦 Paso 13: Exportar modelos para deploy

In [ ]:
# ============================================================
# PASO 13: EXPORTAR A ONNX
# ============================================================
BASE = '/content/drive/MyDrive/fire_severity_project'
import os, json, numpy as np
from datetime import datetime

model.load_state_dict(torch.load(best_path, map_location='cpu'))
model.eval().cpu()

onnx_path = f'{BASE}/models/fire_severity_cnn_v2.onnx'
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, onnx_path,
    input_names=['image'], output_names=['severity_probs'],
    dynamic_axes={'image': {0: 'batch'}}, opset_version=13)

# Verificar
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
test = np.random.randn(1,3,224,224).astype(np.float32)
result = sess.run(None, {'image': test})
print(f'✅ ONNX funciona! Output: {result[0].shape}')

# Tamaños
for name, path in [('CNN .pth', best_path), ('CNN .onnx', onnx_path),
    ('XGBoost', f'{BASE}/models/fire_severity_xgboost_v2.joblib')]:
    if os.path.exists(path):
        print(f'  📏 {name}: {os.path.getsize(path)/(1024*1024):.1f} MB')

# Metadata
meta = {
    'version': 'v2.0',
    'fecha': datetime.now().isoformat(),
    'cnn_accuracy': round(cnn_acc, 2),
    'xgboost_accuracy': round(acc, 2),
    'etiquetado': 'CLIP + HSV + Textura',
    'clases': ['conato','menor','moderado','mayor','catastrofico'],
    'tipos_incendio': ['forestal', 'urbano', 'domestico'],
}
with open(f'{BASE}/models/metadata_v2.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'\n🎉 ¡TODO LISTO!')
print(f'📁 Modelos en: {BASE}/models/')
!ls -lh {BASE}/models/
print(f'\n🚀 Siguiente paso: Subir a Hugging Face Spaces')

---
# ✅ Completado v2!

### Archivos listos en Google Drive:
```
fire_severity_project/models/
├── fire_severity_cnn_v2.onnx        ← Para deploy
├── best_cnn_v2.pth                  ← Backup PyTorch
├── fire_severity_xgboost_v2.joblib  ← Para deploy
└── metadata_v2.json                 ← Info del entrenamiento
```

### 🚀 Siguiente: Crear Hugging Face Space y subir estos modelos